In [ ]:
import pandas as pd
import requests
import time

# ==============================
# 1. 사용자 설정 영역
# ==============================
API_KEY = "CA5D2D47-E9B2-37FC-BA3A-34A9697A1273"

INPUT_EXCEL = r"C:\Users\itwill\Desktop\GIS\260108\서울시_응급실_위치정보.xlsx"
OUTPUT_EXCEL = r"C:\Users\itwill\Desktop\GIS\260108\서울시_응급실_위치정보_좌표.xlsx"

ADDRESS_COLUMN = "주소"   # 엑셀 헤더명 (정확히 일치해야 함)

# ==============================
# 2. VWorld 지오코딩 함수
# ==============================
def geocode_vworld(address: str):
    url = "https://api.vworld.kr/req/address"

    params = {
        "service": "주소",
        "request": "getcoord",
        "address": address,
        "type": "ROAD",      # 도로명 우선
        "format": "json",
        "key": CA5D2D47-E9B2-37FC-BA3A-34A9697A1273
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()
        data = response.json()

        if data["response"]["status"] != "OK":
            return None, None

        point = data["response"]["result"]["point"]
        return float(point["x"]), float(point["y"])

    except Exception:
        return None, None

# ==============================
# 3. 엑셀 처리
# ==============================
df = pd.read_excel(INPUT_EXCEL)

# 결과 컬럼 추가
df["경도"] = None
df["위도"] = None

for idx, address in df[ADDRESS_COLUMN].items():
    if pd.isna(address):
        continue

    x, y = geocode_vworld(str(address))

    df.at[idx, "경도"] = x
    df.at[idx, "위도"] = y

    print(f"[{idx}] {address} → ({x}, {y})")

    # API 과호출 방지
    time.sleep(0.1)

# ==============================
# 4. 결과 저장
# ==============================
df.to_excel(OUTPUT_EXCEL, index=False)
print("완료:", OUTPUT_EXCEL)


[0] 서울특별시 영등포구 63로 10, 여의도성모병원 (여의도동) → (126.936789064, 37.518277576)
[1] 서울특별시 은평구 통일로 1021 (진관동) → (126.916260603, 37.633522358)
[2] 서울특별시 관악구 관악로 242 (봉천동) → (126.956779978, 37.485630494)
[3] 서울특별시 강남구 남부순환로 2649, 베드로병원 (도곡동) → (127.039583954, 37.48559963)
[4] 서울특별시 관악구 남부순환로 1449, 강남힐병원 (신림동) → (126.911648053, 37.481651345)
[5] 서울특별시 강동구 동남로 892 (상일동) → (127.157084732, 37.552045918)
[6] 서울특별시 종로구 새문안로 29 (평동) → (126.967383996, 37.568677243)
[7] 서울특별시 강북구 도봉로 187, 지하1층, 2층~5층 (미아동) → (127.026226562, 37.625418777)
[8] 서울특별시 광진구 능동로 120-1 (화양동) → (127.072787974, 37.540843323)
[9] 서울특별시 송파구 송이로 123, 국립경찰병원 (가락동) → (127.12252148, 37.496386253)
[10] 서울특별시 동대문구 경희대로 23 (회기동) → (127.05074926, 37.59373654)
[11] 서울특별시 구로구 구로동로 148, 고려대부속구로병원 (구로동) → (126.884787386, 37.492116529)
[12] 서울특별시 구로구 경인로 427, 구로성심병원 (고척동) → (126.866338051, 37.499623421)
[13] 서울특별시 중구 을지로 245, 국립중앙의료원 (을지로6가) → (127.005579604, 37.56704024)
[14] 서울특별시 노원구 한글비석로 68, 을지병원 (하계동) → (127.06992832, 37.63646475)
[15] 서울특별시 